In [ ]:
import sys
!{sys.executable} -m pip install langchain-gigachat langchain-chroma langchain-core chromadb

In [ ]:
import json
import os

JSON_PATH = "knowledge_base.json"

if not os.path.exists(JSON_PATH):
    print(f"Файл {JSON_PATH} не найден! Положи его рядом с блокнотом.")
else:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        all_data = json.load(f)
    print(f"Загружено чанков: {len(all_data)}")

In [ ]:
%pip install sentence-transformers

In [ ]:
%pip install langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

docs = [Document(page_content=item["text"], metadata={"source": item["source"]}) for item in all_data]

print("Загружаю локальную модель эмбеддингов...")
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Создаю векторную базу")
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Векторная база готова. Векторов: {vector_store._collection.count()}")

In [ ]:
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GIGACHAT_AUTH_KEY = "здесь ключ"

print("Подключаю GigaChat Ultra...")
llm = GigaChat(
    credentials=GIGACHAT_AUTH_KEY,
    scope="GIGACHAT_API_PERS",
    model="GigaChat-3-Ultra",
    use_api_v2=True,
    verify_ssl_certs=False,
    temperature=0.1
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

system_prompt = """Ты — ИИ-ассистент Корсик компании ПОЛИПЛАСТИК. 
Отвечай вежливо, кратко и строго по делу.
Отвечай на вопрос, используя ТОЛЬКО предоставленный контекст.
Не выдумывай факты.

Если в контексте нет ответа, определи характер вопроса:
1. Если вопрос касается работы, компании или HR-процессов (но ответа нет в базе), напиши ровно одну фразу: "К сожалению, у меня нет этой информации. Я передаю ваш вопрос HR-специалисту."
2. Если вопрос не касается рабочего процесса, является неэтичным, оскорбительным, провокационным или нерелевантным, напиши ровно одну фразу: "К сожалению, данный вопрос не касается рабочего процесса."

Контекст:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

def ask_corsik(question: str) -> dict:
    docs = retriever.invoke(question)
    
    context = "\n\n".join([f"[Источник: {d.metadata['source']}]\n{d.page_content}" for d in docs])
    
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    gigachat_refusal = "Как и любая языковая модель, GigaChat не обладает собственным мнением"
    if gigachat_refusal in answer:
        answer = "К сожалению, запрос некорректен."
    
    sources = list(set([d.metadata['source'] for d in docs]))
    
    escalate = "К сожалению, у меня нет этой информации. Я передаю ваш вопрос HR-специалисту." in answer
    
    return {
        "answer": answer, 
        "sources": sources,
        "escalate": escalate 
    }

print("Модель GigaChat-3-Ultra, RAG-логика и Guardrails обновлены.")

In [ ]:
q1 = "Сколько дней ежегодного оплачиваемого отпуска мне положено?"
res1 = ask_corsik(q1)

print(f"Вопрос: {q1}")
print(f"Ответ Корсика: {res1['answer']}")
print(f"Источники: {res1['sources']}")
print(f"Нужен HR?: {res1['escalate']}\n")
print("-" * 50)

q2 = "Блаблабла"
res2 = ask_corsik(q2)

print(f"Вопрос: {q2}")
print(f"Ответ Корсика: {res2['answer']}")
print(f"Источники: {res2['sources']}")
print(f"Нужен HR?: {res2['escalate']}")

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — HR-аналитик. Проанализируй текст сотрудника.
Верни ответ СТРОГО в формате JSON без лишнего текста и без markdown-разметки:
{{
  "sentiment": "positive" | "neutral" | "negative",
  "tags": ["burnout", "toxic", "problem_with_access", "misunderstanding", "positive"],
  "red_flag": true | false (true, если есть выгорание, токсичность или сильное недовольство)
}}"""),
    ("human", "{text}")
])

def analyze_sentiment(text: str) -> dict:
    chain = sentiment_prompt | llm | StrOutputParser()
    result = chain.invoke({"text": text})

    clean_result = result.replace("```json", "").replace("```", "").strip()
    
    try:
        return json.loads(clean_result)
    except Exception as e:
        return {"sentiment": "unknown", "tags": [], "red_flag": False, "error": str(e), "raw": result}

test_texts = [
    "Все отлично! Коллектив супер, задачи понятные, доступы выдали в первый день.",
    "Я уже устал, а задачи все валятся. Руководитель постоянно орет и не помогает.",
    "Пока ничего не понятно, жду доступ к порталу уже второй день."
]

print("Тестируем анализ настроений:\n")
for text in test_texts:
    print(f"Текст: {text}")
    print(f"Анализ: {analyze_sentiment(text)}\n")
    print("-" * 50)